In [2]:
import torch
from torch import nn


device = "cuda" if torch.cuda.is_available() else "cpu"


class SingleTokenMLP(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(paragraph_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2048, decoder_hidden_dim)
        )

    def forward(self, x):
        return self.net(x)


class EndToEndInverter(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim, prefix_len, decoder_model):
        super().__init__()
        self.prefix_len = prefix_len
        self.m_parallel_mlps = nn.ModuleList(
            [SingleTokenMLP(paragraph_dim, decoder_hidden_dim) for _ in range(prefix_len)]
        )
        self.decoder = decoder_model

    def _prefix(self, paragraph_embs):
        return torch.stack([mlp(paragraph_embs) for mlp in self.m_parallel_mlps], dim=1)

    @torch.no_grad()
    def generate(self, paragraph_embs, tokenizer, max_new_tokens=128):
        paragraph_embs = paragraph_embs.to(next(self.parameters()).device)
        prefix = self._prefix(paragraph_embs).to(dtype=self.decoder.dtype)
        outputs = self.decoder.generate(
            inputs_embeds=prefix,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.2,
        )
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [4]:
from transformers import AutoModel
vanilla_encoder = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
embedding_inverter = EndToEndInverter(paragraph_dim=1024,
                                      decoder_hidden_dim=1024,
                                      prefix_len=64,
                                      decoder_model="Qwen/Qwen3-0.6B")

In [9]:
param_vanilla = sum(p.numel() for p in vanilla_encoder.parameters())
param_mlps =  sum(p.numel() for p in embedding_inverter.parameters())
print(param_mlps/param_vanilla * 100)

45.0894015774828


In [10]:
param_vanilla

595776512

In [11]:
param_mlps

268632064